In [ ]:
###
# セットアップ
###

import sys
import os
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

ROOT_PATH = Path("/content/drive/MyDrive/cnn-hands-on")

if str(ROOT_PATH) not in sys.path:
    sys.path.append(str(ROOT_PATH))

In [ ]:
###
# 犬猫データセットをダウンロードし、画像ファイルとして保存するセル
###

import os
from datasets import load_dataset
from tqdm import tqdm

from google.colab import drive
drive.mount("/content/drive")

# 保存先ディレクトリの作成
data_dir = ROOT_PATH / "data" / "cats_vs_dogs"
cat_dir = os.path.join(data_dir, "Cat")
dog_dir = os.path.join(data_dir, "Dog")
os.makedirs(cat_dir, exist_ok=False)
os.makedirs(dog_dir, exist_ok=False)

print("データセットをダウンロード中")
dataset = load_dataset("microsoft/cats_vs_dogs")

print(f" 画像を {data_dir} に展開中")

# ラベルの対応 (データセットの仕様上、0: Cat, 1: Dog)
label_map = {0: cat_dir, 1: dog_dir}

# trainスプリットのデータを1枚ずつJPGとして保存
for i, item in enumerate(tqdm(dataset["train"])):
    img = item["image"]
    label = item["labels"]
    
    # 保存先のパス
    save_path = os.path.join(label_map[label], f"{i:05d}.jpg")
    
    # ファイルがまだ無ければ保存
    if not os.path.exists(save_path):
        if img.mode != "RGB":
            img = img.convert("RGB")
        img.save(save_path)

print("ダウンロード完了")

In [ ]:
###
# 画像一覧を読み込み、Train / Val / Test に分割したラベルCSVを作成する
###

import os
import pandas as pd
import numpy as np

data_dir = ROOT_PATH / "data" / "cats_vs_dogs"
cat_dir = os.path.join(data_dir, "Cat")
dog_dir = os.path.join(data_dir, "Dog")

data_list = []

# 猫画像 (Label: 0) をリストに追加
for file_name in os.listdir(cat_dir):
    if file_name.endswith(('.jpg', '.jpeg', '.png')):
        rel_path = f"Cat/{file_name}"
        data_list.append({"filepath": rel_path, "label": 0, "class_name": "Cat"})

# 犬画像 (Label: 1) をリストに追加
for file_name in os.listdir(dog_dir):
    if file_name.endswith(('.jpg', '.jpeg', '.png')):
        rel_path = f"Dog/{file_name}"
        data_list.append({"filepath": rel_path, "label": 1, "class_name": "Dog"})

# pandasのデータフレームに変換
df = pd.DataFrame(data_list)
total_len = len(df)

print(f"総画像枚数: {total_len}枚")

# 80:10:10に分割
df = df.sample(frac=1, random_state=61).reset_index(drop=True)

# indexを計算
train_end = int(total_len * 0.80)
val_end = train_end + int(total_len * 0.1)

df['split'] = 'test'
df.loc[:train_end-1, 'split'] = 'train'
df.loc[train_end:val_end-1, 'split'] = 'val'

csv_path = os.path.join(data_dir, "labels.csv")
df.to_csv(csv_path, index=False)

print("データ分割の内訳")
print(df['split'].value_counts())

print("CSVの先頭5行")
display(df.head())